# Run Stepper Demo

This notebook demonstrates setting up the configuration, building the RK4 stepper, running a short integration, and plotting diagnostics (kinetic energy and enstrophy). It is designed to be quick on CPU for exploratory testing.

In [25]:
import sys
sys.path.append(r"c:\Users\C5044892\Documents\Code\QG")

import time
import textwrap
import numpy as np
import matplotlib.pyplot as plt
get_ipython().run_line_magic('matplotlib', 'inline')
import importlib
import model.utils
importlib.reload(model.utils)
import model.core
importlib.reload(model.core)

import jax
import jax.numpy as jnp
from jax.numpy.fft import irfftn

from model.utils.config import Config
from model.run import _params_from_config
from model.core.grid import Grid
from model.core.steppers import build_stepper
from model.core.solver import Solver
from model.diagnostics.diagnostics import compute_ke, compute_enstrophy

In [ ]:
# test config
cfg = Config.from_dict({
    "grid": {"nx": 64, "ny": 64, "Lx": 6.28},
    "timestep": {"dt": 1e-3, "stepper": "RK4"},
    "forcing": {"k_f": 8.0, "k_width": 2.0},
})

params = _params_from_config(cfg)
print('grid params:', params['nx'], params['ny'], 'dt=', params['dt'])


grid params: 64 64 dt= 0.001
{'nx': 64, 'ny': 64, 'Lx': 6.28, 'dt': 0.001, 'beta': 1.0, 'k_f': 8.0, 'k_width': 2.0, 'epsilon': 0.001}


In [27]:

grid = Grid(params)


# initialize spectral state
key = jax.random.PRNGKey(0)
key, k0 = jax.random.split(key)
qh = Solver.pseudo_randomiser(grid, params.get('k_f', 8.0), k0)
state = (qh, key)

print('initialized, qh shape:', qh.shape)

initialized, qh shape: (64, 33)


In [28]:
# 4) Run short integration and record diagnostics
import os
import json
from pathlib import Path

# allow tests to reduce the number of steps via environment variable
nsteps = int(os.environ.get("QG_NSTEPS", "100"))
ke_list = []
enst_list = []
start = time.time()
for i in range(nsteps):
    state = stepper(state, dt=None)
    qh, key = state
    # compute physical fields
    zeta = irfftn(qh, s=(grid.ny, grid.nx)).real
    psih = -qh * grid.invK2
    psi = irfftn(psih, s=(grid.ny, grid.nx)).real

    ke = compute_ke(psi, grid)
    enst = compute_enstrophy(zeta, grid)
    ke_list.append(ke)
    enst_list.append(enst)
    if (i+1) % max(1, nsteps//5) == 0:
        print(f"step {i+1}/{nsteps} ke={ke:.5g} enst={enst:.5g}")

runtime = time.time() - start
print('finished steps in', runtime, 's')

# write a small result file for tests to inspect
out = {
    "nsteps": int(nsteps),
    "runtime": float(runtime),
    "last_ke": float(ke_list[-1]) if ke_list else None,
    "last_enst": float(enst_list[-1]) if enst_list else None,
}
out_path = Path.cwd() / "notebook_run_results.json"
with open(out_path, "w") as f:
    json.dump(out, f)
print('WROTE_RESULTS:', out_path)


TypeError: build_stepper.<locals>.<lambda>() got an unexpected keyword argument 'dt'

In [ ]:
# 5) Plot diagnostics and final fields
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
axs[0].plot(np.arange(1, nsteps+1) * params['dt'], ke_list, label='KE')
axs[0].set_xlabel('time')
axs[0].set_title('Kinetic Energy')
axs[0].legend()

axs[1].plot(np.arange(1, nsteps+1) * params['dt'], enst_list, label='Enstrophy', color='C1')
axs[1].set_xlabel('time')
axs[1].set_title('Enstrophy')
axs[1].legend()

# final vorticity
zeta = irfftn(state[0], s=(grid.ny, grid.nx)).real
c = axs[2].imshow(np.array(zeta), origin='lower', cmap='RdBu')
axs[2].set_title('Final vorticity')
fig.colorbar(c, ax=axs[2])
plt.tight_layout()
plt.show()